In [ ]:
# ════════════════════════════════════════════════════════════
# PIPELINE — Cell 1: CONFIGURATION
# Headshots -> LivePortrait (idle + talking) -> TTS (voice by gender)
#   -> boomerang idle+speaking+idle -> audio (muxed + separate padded).
# Edit the config below, then run cells 2 → 5 in order.
# (LivePortrait needs a GPU; run on Colab. ffmpeg + ffprobe must be on PATH.)
# ════════════════════════════════════════════════════════════
import os, sys, glob, json, subprocess, time, tempfile
from pathlib import Path

# Mount Google Drive automatically when on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print(f'Environment: {"Colab" if IN_COLAB else "local"}')

# ── Folders & driving templates ──────────────────────────────
BASE_DIR         = '/content/drive/MyDrive/talking_head' if IN_COLAB else './assets'
TALKING_TEMPLATE = f'{BASE_DIR}/template_talking.mp4'   # expressive driving video
IDLE_TEMPLATE    = f'{BASE_DIR}/template_idle.mp4'      # subtle "breathing" driving video
HEADSHOTS_FOLDER = f'{BASE_DIR}/headshots'              # person_XX.png / .jpg
ANIMATION_FOLDER = f'{BASE_DIR}/outputs/animations'     # LivePortrait idle + talking
AUDIO_FOLDER     = f'{BASE_DIR}/outputs/audio'          # TTS mp3s
OUTPUT_FOLDER    = f'{BASE_DIR}/outputs/final'          # sequenced + final
MODELS_DIR       = '/content/models'                    # LivePortrait repo (ephemeral)

# ── Headshots and gender (name -> isMale) ────────────────────
HEADSHOTS = {
    'person_01': True,
    'person_02': True,
    'person_03': False,
    'person_04': False,
    'person_05': True,
    'person_06': True,
    'person_07': False,
    'person_08': False,
}

# ── TTS (one shared paragraph; voice depends on isMale) ──────
TTS_TEXT = (
    "Hey there, it's really good to see you. So, here's the deal, I'm here to make "
    "your day a little easier, whether that's answering a quick question or walking "
    "you through something step by step. There's honestly no rush at all, we'll just "
    "take it at whatever pace feels right for you. So go ahead, get comfortable, and "
    "whenever you're ready, just say the word and we'll jump right in."
)
VOICE_FEMALE = 'en-US-AvaMultilingualNeural'      # warm, human
VOICE_MALE   = 'en-US-AndrewMultilingualNeural'   # warm, human
TTS_RATE, TTS_PITCH, TTS_VOLUME = '-4%', '+0Hz', '+0%'

# ── Animation / boomerang params ─────────────────────────────
DRIVING_MULT_TALKING = 1.1     # LivePortrait expressiveness for the talking pass
DRIVING_MULT_IDLE    = 1.0     # subtler for the idle pass
IDLE_DURATION  = 3.0           # seconds of idle at start and end
NORMAL_SEC     = 1.4           # idle boomerang: normal-speed portion
EASE_SRC_SEC   = 0.2           # source secs slowed at each turnaround...
EASE_OUT_SEC   = 0.3           # ...stretched to this long (smoother reversal)
CROSSFADE_SEC  = 0.0           # 0 = hard cut at the idle<->speaking joins
AUDIO_OFFSET   = 3.0           # audio starts here (= idle lead-in)

# ── Make output folders ──────────────────────────────────────
for d in (ANIMATION_FOLDER, AUDIO_FOLDER, OUTPUT_FOLDER):
    os.makedirs(d, exist_ok=True)

# ── Path helpers (used by all cells) ─────────────────────────
def headshot_path(name):
    for ext in ('.png', '.jpg', '.jpeg'):
        p = os.path.join(HEADSHOTS_FOLDER, name + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'No headshot for {name} in {HEADSHOTS_FOLDER}')

def anim_path(name, kind):                 # kind: 'idle' or 'talking'
    return os.path.join(ANIMATION_FOLDER, f'{name}_{kind}.mp4')
def audio_path(name):
    return os.path.join(AUDIO_FOLDER, f'{name}.mp3')
def sequence_path(name):
    return os.path.join(OUTPUT_FOLDER, f'{name}_sequenced.mp4')
def muxed_path(name):
    return os.path.join(OUTPUT_FOLDER, f'{name}_with_audio.mp4')
def padded_audio_path(name):
    return os.path.join(OUTPUT_FOLDER, f'{name}_audio_padded.m4a')

# ── Shared ffprobe / ffmpeg helpers (used by all cells) ──────
def _dur(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                          '-of', 'json', str(path)], capture_output=True, text=True).stdout
    return float(json.loads(out)['format']['duration'])
_duration = _dur   # alias used by the idle-boomerang helper

def _fps(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                          '-show_entries', 'stream=r_frame_rate', '-of', 'json',
                          str(path)], capture_output=True, text=True).stdout
    n, d = json.loads(out)['streams'][0]['r_frame_rate'].split('/')
    return float(n) / float(d)

def _res(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                          '-show_entries', 'stream=width,height', '-of', 'json',
                          str(path)], capture_output=True, text=True).stdout
    s = json.loads(out)['streams'][0]
    return int(s['width']), int(s['height'])

def _ff(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('ffmpeg failed:\n' + (r.stderr or '')[-1500:])

# Shared state filled by later cells.
AUDIO_DUR = {}   # name -> TTS audio duration (= S for the speaking segment)

print(f'Headshots ({len(HEADSHOTS)}): {list(HEADSHOTS)}')
print(f'Templates : talking={TALKING_TEMPLATE}')
print(f'            idle   ={IDLE_TEMPLATE}')
print(f'Outputs   : {ANIMATION_FOLDER} | {AUDIO_FOLDER} | {OUTPUT_FOLDER}')


In [ ]:
# ════════════════════════════════════════════════════════════
# PIPELINE — Cell 2: LIVEPORTRAIT  (idle + talking per headshot)
# Same setup / weights / InsightFace logic as M1_liveportrait_template.
# Renders TWO videos per headshot:
#   {name}_talking.mp4  (driven by TALKING_TEMPLATE, mult DRIVING_MULT_TALKING)
#   {name}_idle.mp4     (driven by IDLE_TEMPLATE,    mult DRIVING_MULT_IDLE)
# ════════════════════════════════════════════════════════════
LP_REPO = Path(MODELS_DIR) / 'liveportrait'
LP_REPO.parent.mkdir(parents=True, exist_ok=True)
HF_REPO = 'KlingTeam/LivePortrait'
WEIGHT_FILES = [
    'liveportrait/base_models/appearance_feature_extractor.pth',
    'liveportrait/base_models/motion_extractor.pth',
    'liveportrait/base_models/spade_generator.pth',
    'liveportrait/base_models/warping_module.pth',
    'liveportrait/retargeting_models/stitching_retargeting_module.pth',
    'liveportrait/landmark.onnx',
]

def _fmt(s):
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}h {m:02d}m {s:02d}s' if h else (f'{m}m {s:02d}s' if m else f'{s}s')
def _step(msg): print(f'\n[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

for t in (TALKING_TEMPLATE, IDLE_TEMPLATE):
    if not os.path.exists(t):
        raise RuntimeError(f'Driving template not found: {t}')

# ── Setup (clone + install) ──────────────────────────────────
if not (LP_REPO / '.setup_done').exists():
    _step('SETUP — cloning LivePortrait + installing deps')
    if not LP_REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/KwaiVGI/LivePortrait', str(LP_REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', str(LP_REPO / 'requirements_base.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'onnxruntime', 'transformers==4.38.0', 'huggingface_hub'], check=True)
    (LP_REPO / '.setup_done').touch()
    print('  Setup done.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Setup already done, skipping.')

# ── Weights (~600 MB, once) ──────────────────────────────────
if not (LP_REPO / '.weights_done').exists():
    _step('WEIGHTS — downloading 6 files from HuggingFace')
    from huggingface_hub import hf_hub_download
    weights_dir = LP_REPO / 'pretrained_weights'
    weights_dir.mkdir(parents=True, exist_ok=True)
    for f in WEIGHT_FILES:
        dest = weights_dir / f
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            print(f'  {Path(f).name} ...', end=' ', flush=True)
            t0 = time.time()
            hf_hub_download(repo_id=HF_REPO, filename=f, local_dir=str(weights_dir))
            print(f'{dest.stat().st_size/1024/1024:.0f} MB in {_fmt(time.time()-t0)}')
        else:
            print(f'  already have {Path(f).name}')
    (LP_REPO / '.weights_done').touch()
else:
    print(f'[{time.strftime("%H:%M:%S")}] Weights already downloaded, skipping.')

# ── InsightFace buffalo_l (det + landmark) ───────────────────
# LivePortrait crops faces with InsightFace, which only needs det_10g.onnx
# + 2d106det.onnx. But FaceAnalysis loads EVERY .onnx in the folder on init,
# so a corrupt/auto-downloaded 1k3d68.onnx breaks startup. Keep the folder
# to exactly these two valid files from the HF weights repo. Idempotent.
from huggingface_hub import hf_hub_download
_buffalo = LP_REPO / 'pretrained_weights' / 'insightface' / 'models' / 'buffalo_l'
_needed  = ['det_10g.onnx', '2d106det.onnx']
def _ok(p): return p.exists() and p.stat().st_size > 1_000_000   # stubs are <1 KB
if _buffalo.exists():
    for p in _buffalo.glob('*.onnx'):       # drop corrupt or unused onnx
        if p.name not in _needed or not _ok(p):
            p.unlink()
_buffalo.mkdir(parents=True, exist_ok=True)
if not all(_ok(_buffalo / n) for n in _needed):
    _step('INSIGHTFACE — downloading buffalo_l det + landmark models')
    for n in _needed:
        if not _ok(_buffalo / n):
            print(f'  {n} ...', end=' ', flush=True)
            hf_hub_download(repo_id=HF_REPO,
                            filename=f'insightface/models/buffalo_l/{n}',
                            local_dir=str(LP_REPO / 'pretrained_weights'))
            print(f'{(_buffalo/n).stat().st_size/1024/1024:.0f} MB')
print(f'[{time.strftime("%H:%M:%S")}] buffalo_l ready: '
      f'{sorted(p.name for p in _buffalo.glob("*.onnx"))}')

# ── Inference helper (one render) ────────────────────────────
# --flag_crop_driving_video crops the template to its face region so the
# motion transfers cleanly to each still headshot.
def run_liveportrait(headshot, template, out_path, multiplier):
    if os.path.exists(out_path):
        print(f'  skip {os.path.basename(out_path)} (exists)')
        return out_path
    work_dir = str(Path(out_path).with_suffix('')) + '_work'
    os.makedirs(work_dir, exist_ok=True)
    cmd = [sys.executable, str(LP_REPO / 'inference.py'),
           '--source', headshot,
           '--driving', template,
           '--output_dir', work_dir,
           '--driving_multiplier', str(multiplier),
           '--flag_crop_driving_video']
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(LP_REPO),
                            env={**os.environ, 'PYTHONUTF8': '1'})
    if result.returncode != 0:
        print(f'  ERROR {os.path.basename(out_path)}\n{result.stderr[-500:]}')
        return None
    mp4s = sorted(glob.glob(os.path.join(work_dir, '**', '*.mp4'), recursive=True),
                  key=os.path.getmtime, reverse=True)
    if mp4s:
        os.replace(mp4s[0], out_path)
        print(f'  {os.path.basename(out_path)}: {_fmt(time.time()-t0)} '
              f'({os.path.getsize(out_path)/1024/1024:.2f} MB)')
        return out_path
    print(f'  WARNING: no mp4 for {os.path.basename(out_path)}')
    return None

# ── Render idle + talking for every headshot ─────────────────
_step(f'INFERENCE — {len(HEADSHOTS)} headshots × (talking + idle)')
total_start = time.time()
for name, isMale in HEADSHOTS.items():
    hs = headshot_path(name)
    print(f'[{time.strftime("%H:%M:%S")}] {name}:')
    run_liveportrait(hs, TALKING_TEMPLATE, anim_path(name, 'talking'), DRIVING_MULT_TALKING)
    run_liveportrait(hs, IDLE_TEMPLATE,    anim_path(name, 'idle'),    DRIVING_MULT_IDLE)
print(f'\n{"="*50}')
print(f'LivePortrait done. Total: {_fmt(time.time()-total_start)}')
print(f'Outputs: {ANIMATION_FOLDER}')


In [ ]:
# ════════════════════════════════════════════════════════════
# PIPELINE — Cell 3: TTS  (one shared paragraph; voice by gender)
# Male headshots -> VOICE_MALE, female -> VOICE_FEMALE (both human voices).
# Saves {name}.mp3 and records its duration as S in AUDIO_DUR[name].
# ════════════════════════════════════════════════════════════
for pkg, mod in [('edge-tts', 'edge_tts'), ('nest_asyncio', 'nest_asyncio')]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
import edge_tts, nest_asyncio, asyncio
nest_asyncio.apply()   # lets asyncio run inside the notebook's event loop

print(f'TTS — {len(HEADSHOTS)} headshots   female={VOICE_FEMALE}   male={VOICE_MALE}')
for name, isMale in HEADSHOTS.items():
    voice = VOICE_MALE if isMale else VOICE_FEMALE
    out   = audio_path(name)

    async def _gen(v=voice, o=out):
        await edge_tts.Communicate(TTS_TEXT, v,
                                   rate=TTS_RATE, pitch=TTS_PITCH, volume=TTS_VOLUME).save(o)
    asyncio.get_event_loop().run_until_complete(_gen())

    if not os.path.exists(out) or os.path.getsize(out) == 0:
        raise RuntimeError(f'edge-tts produced no audio for {name} '
                           '(check the voice id and your internet connection).')
    AUDIO_DUR[name] = _dur(out)
    print(f'  {name}: {"male  " if isMale else "female"}  {voice}  ->  {AUDIO_DUR[name]:.2f}s')

print(f'\nAudio saved to {AUDIO_FOLDER}')
print(f'Durations (S): ' + ', '.join(f'{n}={d:.2f}s' for n, d in AUDIO_DUR.items()))


In [ ]:
# ════════════════════════════════════════════════════════════
# PIPELINE — Cell 4: BOOMERANG SEQUENCE  (idle 3s + speaking S + idle 3s)
# Reuses the M2 logic verbatim:
#   make_boomerang  : eased idle boomerang from {name}_idle.mp4
#   build_speaking  : looped + boomerang talking clip of EXACTLY S = AUDIO_DUR
#   build_sequence  : stitch idle(3) + speaking(S) + idle(3) -> {name}_sequenced.mp4
# ════════════════════════════════════════════════════════════
def _eased_forward(src, out, half, fps, e_src, e_out):
    """First `half` seconds of src, with the last e_src s slowed to e_out s."""
    half = float(half)
    normal = half - e_src
    if normal > 0 and e_src > 0 and e_out > 0:
        factor = e_out / e_src                      # >1 = slower
        fc = (f'[0:v]trim=0:{normal:.6f},setpts=PTS-STARTPTS[a];'
              f'[0:v]trim={normal:.6f}:{half:.6f},setpts={factor:.6f}*(PTS-STARTPTS)[b];'
              f'[a][b]concat=n=2:v=1,fps={fps:.6f}[v]')
    else:                                            # too short to ease — plain clip
        fc = f'[0:v]trim=0:{half:.6f},setpts=PTS-STARTPTS,fps={fps:.6f}[v]'
    _ff(['ffmpeg', '-y', '-i', str(src), '-filter_complex', fc, '-map', '[v]', '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', str(out)])
    return out

def _boomerang(fwd, out, fps, drop_first=False):
    """forward + reverse(forward), minus the duplicate turnaround frame.
    drop_first also removes the very first frame (for a seamless join after a
    previous piece that already ends on this same frame)."""
    base = ('[0:v]split[x][y];'
            '[y]reverse,trim=start_frame=1,setpts=PTS-STARTPTS[r];'
            f'[x][r]concat=n=2:v=1,fps={fps:.6f}')
    fc = (base + f',trim=start_frame=1,setpts=PTS-STARTPTS,fps={fps:.6f}[v]') if drop_first \
         else (base + '[v]')
    _ff(['ffmpeg', '-y', '-i', str(fwd), '-filter_complex', fc, '-map', '[v]', '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', str(out)])
    return out

def make_boomerang(src, out, normal_sec, ease_src_sec=0.0, ease_out_sec=0.0):
    """forward (with optional slowed tail) + reverse(forward), video only (no audio)."""
    src, out = str(src), str(out)
    if not os.path.exists(src):
        raise FileNotFoundError(f'Input not found: {src}')
    os.makedirs(os.path.dirname(out) or '.', exist_ok=True)
    fps = _fps(src)
    eased = ease_src_sec > 0 and ease_out_sec > 0
    need  = normal_sec + (ease_src_sec if eased else 0.0)
    src_dur = _duration(src)
    if src_dur < need:
        print(f'  WARNING: source is {src_dur:.2f}s (< {need:.2f}s needed) — clamping.')
        normal_sec = max(0.0, src_dur - (ease_src_sec if eased else 0.0))

    if eased:
        factor = ease_out_sec / ease_src_sec          # >1 = slower
        fwd = (f'[0:v]trim=0:{normal_sec},setpts=PTS-STARTPTS[h];'
               f'[0:v]trim={normal_sec}:{normal_sec + ease_src_sec},'
               f'setpts={factor:.6f}*(PTS-STARTPTS)[t];'
               f'[h][t]concat=n=2:v=1,setpts=PTS-STARTPTS[fwd]')
        half_out = normal_sec + ease_out_sec
    else:
        fwd = f'[0:v]trim=0:{normal_sec},setpts=PTS-STARTPTS[fwd]'
        half_out = normal_sec

    fc = f'{fwd};[fwd]split[a][b];[b]reverse[r];[a][r]concat=n=2:v=1,fps={fps:.6f}[v]'
    cmd = ['ffmpeg', '-y', '-i', src, '-filter_complex', fc,
           '-map', '[v]', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', out]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('ffmpeg failed:\n' + r.stderr[-1500:])
    print(f'  done -> {out}\n         {_duration(out):.2f}s (target {2*half_out:.2f}s), '
          f'{fps:.2f} fps, no audio')
    return out

def build_speaking(src, out, S, e_src, e_out):
    src, out = str(src), str(out)
    if not os.path.exists(src):
        raise FileNotFoundError(f'Input not found: {src}')
    os.makedirs(os.path.dirname(out) or '.', exist_ok=True)
    V, fps = _dur(src), _fps(src)
    tmp = tempfile.mkdtemp()
    posix = lambda p: str(p).replace('\\', '/')

    # Estimated unit (half-video eased boomerang) length, for the loop count.
    half_unit = V / 2.0
    L = (V + 2.0 * (e_out - e_src) - 1.0 / fps) if half_unit > e_src else (V - 1.0 / fps)
    N = max(0, int(S // L)) if L > 0 else 0
    R = S - N * L

    pieces = []
    if N >= 1:
        fwd_u = _eased_forward(src, f'{tmp}/fwd_u.mp4', half_unit, fps, e_src, e_out)
        unit_full  = _boomerang(fwd_u, f'{tmp}/unit_full.mp4',  fps, drop_first=False)
        pieces.append(unit_full)
        if N >= 2:
            unit_nodup = _boomerang(fwd_u, f'{tmp}/unit_nodup.mp4', fps, drop_first=True)
            pieces += [unit_nodup] * (N - 1)
        if R > 0.3:                                  # leftover worth its own boomerang
            fwd_r = _eased_forward(src, f'{tmp}/fwd_r.mp4', R / 2.0, fps, e_src, e_out)
            pieces.append(_boomerang(fwd_r, f'{tmp}/rem.mp4', fps, drop_first=True))
    else:                                            # S <= unit: one eased boomerang
        R = S
        fwd_r = _eased_forward(src, f'{tmp}/fwd_r.mp4', S / 2.0, fps, e_src, e_out)
        pieces.append(_boomerang(fwd_r, f'{tmp}/rem.mp4', fps, drop_first=False))

    # Concat all pieces (identical codec/params -> demuxer is safe).
    listf = f'{tmp}/list.txt'
    with open(listf, 'w') as f:
        for p in pieces:
            f.write(f"file '{posix(p)}'\n")
    concat_tmp = f'{tmp}/concat.mp4'
    _ff(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', listf, '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', '-r', f'{fps:.6f}', concat_tmp])

    # Stretch the assembled clip to land on exactly S (absorbs dropped frames).
    D = _dur(concat_tmp)
    factor = S / D
    fc = f'[0:v]setpts={factor:.6f}*PTS,fps={fps:.6f}[v]'
    _ff(['ffmpeg', '-y', '-i', concat_tmp, '-filter_complex', fc, '-map', '[v]', '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', out])

    final = _dur(out)
    print(f'  V={V:.2f}s | unit~{L:.2f}s | S={S:.2f}s  ->  {N} seamless unit(s) '
          f'+ {R:.2f}s remainder boomerang')
    print(f'  pre-stretch {D:.2f}s -> stretched x{factor:.4f} -> {final:.2f}s, {fps:.2f} fps, no audio')
    print(f'  done -> {out}')
    return out

def build_sequence(idle_clip, speaking_clip, out, idle_dur, crossfade):
    for f in (idle_clip, speaking_clip):
        if not os.path.exists(f):
            raise FileNotFoundError(f'Input not found: {f}')
    os.makedirs(os.path.dirname(out) or '.', exist_ok=True)
    fps  = _fps(speaking_clip)
    W, H = _res(speaking_clip)
    tmp  = tempfile.mkdtemp()

    # Idle -> stretched to exactly idle_dur, normalized to speaking fps/res.
    k = idle_dur / _dur(idle_clip)
    idle_seg = f'{tmp}/idle_seg.mp4'
    _ff(['ffmpeg', '-y', '-i', str(idle_clip), '-filter_complex',
         f'[0:v]setpts={k:.6f}*PTS,fps={fps:.6f},scale={W}:{H},setsar=1,format=yuv420p[v]',
         '-map', '[v]', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', idle_seg])

    # Speaking -> normalized only (its length IS S).
    speak_seg = f'{tmp}/speak_seg.mp4'
    _ff(['ffmpeg', '-y', '-i', str(speaking_clip), '-filter_complex',
         f'[0:v]fps={fps:.6f},scale={W}:{H},setsar=1,format=yuv420p[v]',
         '-map', '[v]', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', speak_seg])
    S = _dur(speak_seg)

    if crossfade and crossfade > 0:                      # soft blends at the 2 joins
        cf = float(crossfade)
        off1 = idle_dur - cf
        off2 = idle_dur + S - 2 * cf
        fc = (f'[0:v]settb=AVTB[a];[1:v]settb=AVTB[b];[2:v]settb=AVTB[c];'
              f'[a][b]xfade=transition=fade:duration={cf:.6f}:offset={off1:.6f}[ab];'
              f'[ab][c]xfade=transition=fade:duration={cf:.6f}:offset={off2:.6f}[v]')
        mode = f'crossfade {cf:.2f}s'
    else:                                                # hard cuts, exact 6+S
        fc = '[0:v][1:v][2:v]concat=n=3:v=1[v]'
        mode = 'hard cut'

    _ff(['ffmpeg', '-y', '-i', idle_seg, '-i', speak_seg, '-i', idle_seg,
         '-filter_complex', fc, '-map', '[v]', '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', '-r', f'{fps:.6f}', str(out)])

    total = _dur(out)
    print(f'  timeline ({mode}):')
    print(f'    0.00 -> {idle_dur:5.2f} s   idle')
    print(f'    {idle_dur:5.2f} -> {idle_dur+S:5.2f} s   speaking (S={S:.2f}s)')
    print(f'    {idle_dur+S:5.2f} -> {2*idle_dur+S:5.2f} s   idle')
    print(f'  done -> {out}\n         {total:.2f}s, {W}x{H} @ {fps:.2f} fps, no audio')
    return out

# ── Build the full idle->speaking->idle clip for every headshot ──
_seq_tmp = tempfile.mkdtemp()
print(f'SEQUENCE — {len(HEADSHOTS)} headshots\n')
for name in HEADSHOTS:
    idle_v, talk_v = anim_path(name, 'idle'), anim_path(name, 'talking')
    S = AUDIO_DUR[name]
    idle_clip  = os.path.join(_seq_tmp, f'{name}_idle.mp4')
    speak_clip = os.path.join(_seq_tmp, f'{name}_speak.mp4')
    print(f'[{name}]  idle boomerang + speaking({S:.2f}s) + stitch')
    make_boomerang(idle_v, idle_clip, NORMAL_SEC, EASE_SRC_SEC, EASE_OUT_SEC)
    build_speaking(talk_v, speak_clip, S, EASE_SRC_SEC, EASE_OUT_SEC)
    build_sequence(idle_clip, speak_clip, sequence_path(name), IDLE_DURATION, CROSSFADE_SEC)
    print()

print(f'Sequenced videos -> {OUTPUT_FOLDER}')


In [ ]:
# ════════════════════════════════════════════════════════════
# PIPELINE — Cell 5: AUDIO  (muxed mp4 + separate padded audio)
# For each headshot, produce BOTH:
#   • muxed  : audio baked onto the video at AUDIO_OFFSET (3s)
#              -> {name}_with_audio.mp4   (one-file convenience export)
#   • padded : audio with 3s silence front + back, so it is the SAME length
#              as the 6+S video and lines up 1:1 (kept as a separate asset)
#              -> {name}_audio_padded.m4a
# ════════════════════════════════════════════════════════════
print(f'AUDIO — {len(HEADSHOTS)} headshots   (mux + separate padded)\n')
for name in HEADSHOTS:
    video = sequence_path(name)
    audio = audio_path(name)
    if not (os.path.exists(video) and os.path.exists(audio)):
        print(f'  skip {name} (missing video or audio)')
        continue

    total    = _dur(video)                       # = 6 + S
    delay_ms = int(round(AUDIO_OFFSET * 1000))

    # 1) Muxed: video copied as-is, audio delayed to start at AUDIO_OFFSET.
    _ff(['ffmpeg', '-y', '-i', video, '-i', audio,
         '-filter_complex', f'[1:a]adelay=delays={delay_ms}:all=1[a]',
         '-map', '0:v', '-map', '[a]',
         '-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k', muxed_path(name)])

    # 2) Separate padded audio: 3s silence + speech + 3s silence (= total).
    #    adelay pads the front, apad pads the tail, -t trims to the video length.
    _ff(['ffmpeg', '-y', '-i', audio,
         '-af', f'adelay={delay_ms}:all=1,apad', '-t', f'{total:.3f}',
         '-c:a', 'aac', '-b:a', '192k', padded_audio_path(name)])

    pad_dur = _dur(padded_audio_path(name))
    print(f'  {name}: muxed -> {os.path.basename(muxed_path(name))}  |  '
          f'padded -> {os.path.basename(padded_audio_path(name))} '
          f'({pad_dur:.2f}s, video {total:.2f}s)')

print(f'\nFinal outputs -> {OUTPUT_FOLDER}')
print('  {name}_sequenced.mp4    (silent idle+speaking+idle)')
print('  {name}_with_audio.mp4   (audio muxed at 3s)')
print('  {name}_audio_padded.m4a (separate, same length as the video)')
